# Project 3: Semantic Segmentation

#### Maximum Points: 100

This project evaluates your understanding and implementation of a semantic segmentation pipeline. The overall project is divided into several sections, with specific marks assigned to each.


**Mark Distribution**:

<div align = "center">
    <table>
        <tr><td><h3>No.</h3></td> <td><h3>Section</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1.1.</h3></td> <td><h3> Dataset Class</h3></td> <td><h3>15</h3></td> </tr>
        <tr><td><h3>1.2</h3></td> <td><h3> Visualize dataset</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2.</h3></td> <td><h3>  Define Evaluation Metrics</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>3.</h3></td> <td><h3>  Model Definition</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>4.</h3></td> <td><h3>  Train</h3></td> <td><h3>15</h3></td> </tr>
        <tr><td><h3>5.</h3></td> <td><h3>  Inference</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>6.</h3></td> <td><h3> mIoU Score</h3></td> <td><h3>30</h3></td> </tr>
    </table>
        
</div>

---

Breakdown of mIoU Score on Test Dataset **(30 Points)**:

The mIoU Score section evaluates the quality of your segmentation model. Points are awarded based on the achieved mIoU (Mean Intersection over Union) score on the test dataset, as shown below:

<div align="center">
    <table>
        <tr><td><h3>No.</h3></td> <td><h3>mIoU Score Range</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1.</h3></td> <td><h3>>=65% and < 70%</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2.</h3></td> <td><h3>>= 70% and < 73%</h3></td> <td><h3>20</h3></td> </tr>
        <tr><td><h3>3.</h3></td> <td><h3>>= 73%</h3></td><td><h3>30</h3></td> </tr>
    </table>
</div>


---
<h2 style = "color: green;">Dataset Description </h2>

For this Semantic Segmentation Project, you will be working on the **[ICPR Retinal Blood Vessels Dataset](https://www.dropbox.com/scl/fi/tscgh3pxwzfvesnu6l6uv/icpr_prepared.zip?rlkey=8oay8sod3jc1hvwhgqvylaefr&st=udj92wmp&dl=1).**


The dataset consists of **268** train images and **112** test images in 2 classes (**veins** and **background**).

There is no separate validation dataset, you can use the given test dataset for validation.

The dataset is structured as follows:

```
icpr_prepared/
├── train_images/
│   ├── image_100.tif
│   ├── image_101.tif
│   └── ...
├── train_labels/
│   ├── image_100.tif
│   ├── image_101.tif
│   └── ...
├── test_images/
│   ├── image_121.tif
│   ├── image_122.tif
│   └── ...
└── test_labels/
    ├── image_121.tif
    ├── image_122.tif
    └── ...

```
---
Images and their corresponding binary masks are of `.tif` file format and are named with a unique `ImageId` as the filename.

Few samples are shown below:

<img src="https://www.dropbox.com/scl/fi/h0z30ptb8o6cyppfs8h4w/segformer_data_viz_grid_2.png?rlkey=tssqlts8o4kea3kf1um3tijjs&st=0sxunbha&dl=1" width="800" height="800">


**The notebook is divided into multiple grading sections. You have to write code, as mentioned for each section.  For other helper functions, you can write `.py` files and import them in the notebook. You have to submit the notebook along with `.py` files. Your submitted code must be runnable without any bug.**

<h2 style = "color: green;">1. Data Exploration </h2>

In this section, you have to write your custom dataset class and visualize a few images (minimum five images) along with their mask overlayed.

<h2 style = "color: green;">1.1. Dataset Class [15 Points]</h2>

**In this sub-section, write your custom dataset class.**

**For example:**

```python
class SemSegDataset(Dataset):
    """ Generic Dataset class for semantic segmentation datasets.

        Arguments:
            train_image: path to input train images
            train_mask: path to input train mask labels
            train_tfms: transformations (if any)
            label_colors_list: mapping of label value to color code
            classes_to_train: number of classes

            Names of images in the images_folder and masks_folder should be the same for the same samples.
    """
```

In [ ]:
from dataclasses import dataclass

from util import *

In [ ]:
@dataclass
class SegmentationDataset(TorchDataset):
    images_root_folder: str
    masks_root_folder: str
    image_transforms: A.Compose
    image_filenames: np.ndarray
    has_masks: bool

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        image_filename = self.image_filenames[idx]
        image = cv2.cvtColor(cv2.imread(f'{self.images_root_folder}{image_filename}'), cv2.COLOR_BGR2RGB)
        if self.has_masks:
            mask_stem = Path(image_filename).stem
            mask_path = next(Path(self.masks_root_folder).glob(f'{mask_stem}.*'))
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            transformed = self.image_transforms(image=image, mask=mask)
            return transformed['image'], transformed['mask']
        transformed = self.image_transforms(image=image)
        return transformed['image']

<h2 style = "color: green;">1.2. Visualize dataset [10 Points]</h2>


**In this sub-section,  you have to plot a few images and its mask.**

**For example:**

---

<img src="https://www.dropbox.com/scl/fi/hrkxc7vlziawumy7pwfbt/grid_22.png?rlkey=d9jrybjc993qtbzngekj1sseu&st=7rsy2rik&dl=1">

---

In [ ]:
def plot_val_images_and_masks(num_images_to_visualize=3):
    val_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.val_transforms,
        image_filenames=env.fetch_val_filenames(),
        has_masks=True,
    )

    fig, axes = plt.subplots(num_images_to_visualize, 2, figsize=(10, 5 * num_images_to_visualize))
    axes[0, 0].set_title('Image')
    axes[0, 1].set_title('Mask')

    for i in range(num_images_to_visualize):
        image_tensor, mask_tensor = val_dataset[i]
        image = gpu_image_tensor_to_numpy_array(image_tensor)
        colored_mask = gpu_mask_tensor_to_colored_mask_numpy_array(mask_tensor)

        axes[i, 0].imshow(image)
        axes[i, 0].axis('off')
        axes[i, 1].imshow(colored_mask)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close()

env = local_env
config = Config(training=False)
plot_val_images_and_masks()

<h2 style = "color: green;">2. Define Evaluation Metrics [10 Points]</h2>


This project is evaluated on the mean <a href='https://learnopencv.com/intersection-over-union-iou-in-object-detection-and-segmentation/'>IoU</a>.

Mean Intersection over Union (mIoU) is a metric that measures the average overlap between predicted and actual objects in an image or set of images. It's calculated by averaging the Intersection over Union (IoU) of each class in an image.

The following illustration helps in better understanding the IoU:

<img src = "https://www.dropbox.com/scl/fi/e67ri02rqou5oeazdh3rb/intersection-over-union-iou.jpg?rlkey=x5fi1ke6svoas4rh7mfxn3ro7&st=peahed63&dl=1" >

**In this section, you have to implement the mIoU evaluation metric.**

In [ ]:
def val_mean_iou(pred_masks):
    val_filenames = env.fetch_val_filenames()
    gt_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.test_transforms,
        image_filenames=val_filenames,
        has_masks=True,
    )

    # Accumulators for global mIoU (sum intersection/union across all images, then divide)
    global_intersection = np.zeros(2)
    global_union = np.zeros(2)

    # Per-image mIoU list
    ious_per_image = []

    for i in range(len(gt_dataset)):
        _, gt_mask = gt_dataset[i]
        gt_mask = gt_mask.numpy() if isinstance(gt_mask, torch.Tensor) else gt_mask
        pred_mask = pred_masks[i].cpu().numpy() if isinstance(pred_masks[i], torch.Tensor) else np.array(pred_masks[i])

        # Resize pred to GT dimensions if they differ
        if pred_mask.shape != gt_mask.shape:
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (gt_mask.shape[1], gt_mask.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)

        # Binarize both masks
        gt_binary = (gt_mask > 0).astype(np.uint8)
        pred_binary = (pred_mask > 0).astype(np.uint8)

        # Per-class IoU (background=0, vessel=1)
        class_ious = []
        for c in range(2):
            gt_c = (gt_binary == c)
            pred_c = (pred_binary == c)
            intersection = np.logical_and(gt_c, pred_c).sum()
            union = np.logical_or(gt_c, pred_c).sum()
            global_intersection[c] += intersection
            global_union[c] += union
            iou = intersection / (union + 1e-6)
            class_ious.append(iou)

        ious_per_image.append(np.mean(class_ious))

    per_image_miou = np.mean(ious_per_image)
    global_class_ious = global_intersection / (global_union + 1e-6)
    global_miou = np.mean(global_class_ious)

    print(f'Per-image mIoU: {per_image_miou:.4f}')
    print(f'Global mIoU:    {global_miou:.4f}')
    return global_miou

<h2 style = "color: green;">3. Model [10 Points]</h2>

**In this section, you have to define your model.**

In [ ]:
class RetinaSegModel(nn.Module):
    def __init__(self, saved_model_weights=None):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name=config.encoder_name,
            encoder_weights=config.encoder_weights,
            in_channels=3,
            classes=config.num_classes,
        )

        if saved_model_weights is not None:
            saved_model_weights = torch.load(saved_model_weights, weights_only=True, map_location='cpu')
            self.model.load_state_dict(saved_model_weights)

        self.model.to(env.device)

    def forward(self, x):
        return self.model(x)

<h2 style = "color: green;">4. Train & Inference</h2>

- **In this section, you have to train the model and infer on sample data.**


- **You can write your trainer class in this section.**


- **If you are using any loss function other than PyTorch standard loss function, you have to define in this section.**


- **This section should also have optimizer and LR-schedular (if using) details.**

<h2 style = "color: green;">4.1. Train [15 Points]</h2>


**Write your training code in this section.**


**This section must contain training plots (use matplotlib or share tensorboard scalars logs).**

**You must have to plot the following:**
- **train loss and validation loss**


- **train accuracy and validation accuracy**


- **Mean Intersection over Union (mIoU) of these classes on validation data.**

**An example of training plots is shown below:**

---

<img src='https://www.dropbox.com/scl/fi/c09gejfn8ivhapp2ngott/loss.png?rlkey=6pennpz1959plb84jlfjytw4p&st=e6ppi338&dl=1'>

---

<img src='https://www.dropbox.com/scl/fi/9h83bbzkifn2kon0gu3wi/accuracy.png?rlkey=v0q17rmi1djljny7ykr0ydk7b&st=zxze36ro&dl=1'>

---

<img src='https://www.dropbox.com/scl/fi/0pq7ivltoeyp1mlkv85oh/miou.png?rlkey=g4iuqvbwih9ld6efvayz5lgjj&st=5bpihwaz&dl=1'>



In [ ]:
class RetinaSegLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        targets = targets.unsqueeze(1).float()
        targets = (targets > 0.5).float()
        return self.loss(logits, targets)

In [ ]:
def train_one_epoch(start_time, model, loader, optimizer, loss_function, scaler, scheduler, metrics):
    model.train()
    metrics.reset()
    running_loss = 0.0

    num_batches = num_batches_from_loader(loader)
    for batch_number, (x, y) in enumerate(loader):
        print(f't={time.time() - start_time:.2f}: Loading training batch {batch_number + 1}/{num_batches}')

        x = x.to(env.device, non_blocking=True)
        y = y.to(env.device, non_blocking=True)

        if config.verbose or batch_number == 0:
            allocated = torch.cuda.memory_allocated(env.device) / 1024**3
            reserved = torch.cuda.memory_reserved(env.device) / 1024**3
            print(f'Memory allocated={allocated:.2f} GiB, reserved={reserved:.2f} GiB')
            print(f'First image with overlayed ground-truth mask:')
            visualize_mask_overlayed_over_image(x[0], (y[0] > 0).long())

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=config.use_amp):
            logits = model(x)
            if config.verbose or batch_number == 0:
                logits_0_mask = (logits[0, 0] > 0).long()
                print(f'First image with overlayed prediction mask:')
                visualize_mask_overlayed_over_image(x[0], logits_0_mask)
            loss = loss_function(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        batch_size = x.size(0)
        running_loss += loss.item() * batch_size
        metrics.update(logits.detach(), y)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss, metrics.compute()

In [ ]:
@torch.no_grad()
def validate_one_epoch(start_time, model, loader, loss_function, metrics):
    model.eval()
    metrics.reset()
    running_loss = 0.0
    pred_masks = []

    num_batches = num_batches_from_loader(loader)
    for batch_number, (x, y) in enumerate(loader):
        print(f'v={time.time() - start_time:.2f}: Loading validation batch {batch_number + 1}/{num_batches}')

        x = x.to(env.device, non_blocking=True)
        y = y.to(env.device, non_blocking=True)

        if config.verbose or batch_number == 0:
            print(f'First image with overlayed ground-truth mask:')
            visualize_mask_overlayed_over_image(x[0], (y[0] > 0).long())

        with torch.amp.autocast('cuda', enabled=config.use_amp):
            logits = model(x)
            if config.verbose or batch_number == 0:
                logits_0_mask = (logits[0, 0] > 0).long()
                print(f'First image with overlayed prediction mask:')
                visualize_mask_overlayed_over_image(x[0], logits_0_mask)
            loss = loss_function(logits, y)

        batch_size = x.size(0)
        running_loss += loss.item() * batch_size
        metrics.update(logits, y)
        batch_masks = (logits[:, 0] > 0).long().cpu()
        pred_masks.extend(batch_masks)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss, metrics.compute(), pred_masks

In [ ]:
def fullsize_validate(pred_masks, fullsize_gt_dataset):
    global_correct = 0
    global_pixels = 0
    global_vessel_inter = 0
    global_vessel_union = 0
    global_bg_inter = 0
    global_bg_union = 0

    per_image_accuracies = []
    per_image_vessel_ious = []
    per_image_bg_ious = []
    per_image_mean_ious = []

    for i in range(len(fullsize_gt_dataset)):
        _, gt_mask = fullsize_gt_dataset[i]
        gt_mask = gt_mask.numpy() if isinstance(gt_mask, torch.Tensor) else gt_mask

        pred_mask = pred_masks[i].cpu().numpy() if isinstance(pred_masks[i], torch.Tensor) else np.array(pred_masks[i])

        if pred_mask.shape != gt_mask.shape:
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (gt_mask.shape[1], gt_mask.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)

        gt_binary = (gt_mask > 0).astype(np.uint8)
        pred_binary = (pred_mask > 0).astype(np.uint8)

        # Accuracy
        correct = (pred_binary == gt_binary).sum()
        pixels = gt_binary.size
        global_correct += correct
        global_pixels += pixels
        per_image_accuracies.append(correct / (pixels + 1e-6))

        # Vessel IoU (class=1)
        gt_v = (gt_binary == 1)
        pred_v = (pred_binary == 1)
        v_inter = np.logical_and(gt_v, pred_v).sum()
        v_union = np.logical_or(gt_v, pred_v).sum()
        global_vessel_inter += v_inter
        global_vessel_union += v_union
        img_vessel_iou = v_inter / (v_union + 1e-6)
        per_image_vessel_ious.append(img_vessel_iou)

        # Background IoU (class=0)
        gt_bg = (gt_binary == 0)
        pred_bg = (pred_binary == 0)
        bg_inter = np.logical_and(gt_bg, pred_bg).sum()
        bg_union = np.logical_or(gt_bg, pred_bg).sum()
        global_bg_inter += bg_inter
        global_bg_union += bg_union
        img_bg_iou = bg_inter / (bg_union + 1e-6)
        per_image_bg_ious.append(img_bg_iou)

        per_image_mean_ious.append((img_vessel_iou + img_bg_iou) / 2.0)

    global_accuracy = global_correct / (global_pixels + 1e-6)
    global_vessel_iou = global_vessel_inter / (global_vessel_union + 1e-6)
    global_bg_iou = global_bg_inter / (global_bg_union + 1e-6)
    global_mean_iou = (global_vessel_iou + global_bg_iou) / 2.0

    return {
        'global_accuracy': global_accuracy,
        'per_image_accuracy': float(np.mean(per_image_accuracies)),
        'global_vessel_iou': global_vessel_iou,
        'global_bg_iou': global_bg_iou,
        'global_mean_iou': global_mean_iou,
        'per_image_vessel_iou': float(np.mean(per_image_vessel_ious)),
        'per_image_bg_iou': float(np.mean(per_image_bg_ious)),
        'per_image_mean_iou': float(np.mean(per_image_mean_ious)),
    }

In [ ]:
def train(
        start_epoch = 1,
        saved_model_weights = None,
):
    start_time = time.time()
    print('t=0: Starting data prep and model loading')

    run = wandb.init(
        project='retina-segmentation',
        name=f'run={int(start_time)}',
        config={
            'batch_size': config.batch_size,
            'starting_learning_rate': config.starting_learning_rate,
            'max_epochs': config.max_epochs,
            'patience': config.patience,
            'seed': config.seed,
            'model': 'UNet++',
            'encoder': config.encoder_name,
        },
    )

    train_filenames = env.fetch_train_filenames()
    val_filenames = env.fetch_val_filenames()

    model = RetinaSegModel(saved_model_weights=saved_model_weights)

    wandb.watch(model, log='gradients', log_freq=100)

    train_dataset = SegmentationDataset(
        images_root_folder=env.train_images_folder,
        masks_root_folder=env.train_labels_folder,
        image_transforms=config.train_transforms,
        image_filenames=train_filenames,
        has_masks=True,
    )
    val_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.val_transforms,
        image_filenames=val_filenames,
        has_masks=True,
    )
    fullsize_gt_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.test_transforms,
        image_filenames=val_filenames,
        has_masks=True,
    )

    train_loader = create_dataloader(train_dataset, shuffle=True)
    val_loader = create_dataloader(val_dataset, shuffle=False)

    loss_function = RetinaSegLoss()
    optimizer = TorchOptimizers.AdamW(model.parameters(), lr=config.starting_learning_rate)
    scaler = torch.amp.GradScaler('cuda', enabled=config.use_amp)
    scheduler = None

    train_metrics = MetricsAccumulator()
    val_metrics = MetricsAccumulator()

    best_fullsize_global_mean_iou = float('-inf')
    best_epoch = -1

    history = {
        'train_loss': [], 'train_accuracy': [], 'train_vessel_iou': [], 'train_bg_iou': [], 'train_mean_iou': [],
        'val_loss': [], 'val_accuracy': [], 'val_vessel_iou': [], 'val_bg_iou': [], 'val_mean_iou': [],
        'fullsize_global_accuracy': [], 'fullsize_per_image_accuracy': [],
        'fullsize_global_vessel_iou': [], 'fullsize_global_bg_iou': [], 'fullsize_global_mean_iou': [],
        'fullsize_per_image_vessel_iou': [], 'fullsize_per_image_bg_iou': [], 'fullsize_per_image_mean_iou': [],
        'best': {},
    }

    training_start_time = time.time()
    print(f't={training_start_time - start_time:.2f}: Started training')
    print(f'Config: {config}')

    torch.manual_seed(config.seed)

    epochs_since_best = 0

    try:
        for epoch in range(start_epoch, config.max_epochs + 1):
            epoch_start_time = time.time()
            print(f't={epoch_start_time - start_time:.2f}: Starting epoch {epoch}/{config.max_epochs}. Early stopping in {config.patience - epochs_since_best} epochs.')

            train_loss, train_m = train_one_epoch(epoch_start_time, model, train_loader, optimizer, loss_function, scaler, scheduler, train_metrics)

            if env.device == 'cuda':
                torch.cuda.empty_cache()

            val_loss, val_m, pred_masks = validate_one_epoch(epoch_start_time, model, val_loader, loss_function, val_metrics)

            fullsize_m = fullsize_validate(pred_masks, fullsize_gt_dataset)

            history['train_loss'].append(train_loss)
            history['train_accuracy'].append(train_m['accuracy'])
            history['train_vessel_iou'].append(train_m['vessel_iou'])
            history['train_bg_iou'].append(train_m['bg_iou'])
            history['train_mean_iou'].append(train_m['mean_iou'])

            history['val_loss'].append(val_loss)
            history['val_accuracy'].append(val_m['accuracy'])
            history['val_vessel_iou'].append(val_m['vessel_iou'])
            history['val_bg_iou'].append(val_m['bg_iou'])
            history['val_mean_iou'].append(val_m['mean_iou'])

            for key in fullsize_m:
                history[f'fullsize_{key}'].append(fullsize_m[key])

            print(f'================ Epoch {epoch:03d} stats ==================')
            print(f'train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}')
            print(f'train_acc: {train_m["accuracy"]:.4f}  val_acc: {val_m["accuracy"]:.4f}')
            print(f'train_vessel_iou: {train_m["vessel_iou"]:.4f}  val_vessel_iou: {val_m["vessel_iou"]:.4f}')
            print(f'train_bg_iou: {train_m["bg_iou"]:.4f}  val_bg_iou: {val_m["bg_iou"]:.4f}')
            print(f'train_mean_iou: {train_m["mean_iou"]:.4f}  val_mean_iou: {val_m["mean_iou"]:.4f}')
            print(f'fullsize_global_mean_iou: {fullsize_m["global_mean_iou"]:.4f}  fullsize_per_image_mean_iou: {fullsize_m["per_image_mean_iou"]:.4f}')
            print('===================================================')

            wandb.log({
                'train_loss': train_loss,
                'train_accuracy': train_m['accuracy'],
                'train_vessel_iou': train_m['vessel_iou'],
                'train_bg_iou': train_m['bg_iou'],
                'train_mean_iou': train_m['mean_iou'],
                'val_loss': val_loss,
                'val_accuracy': val_m['accuracy'],
                'val_vessel_iou': val_m['vessel_iou'],
                'val_bg_iou': val_m['bg_iou'],
                'val_mean_iou': val_m['mean_iou'],
                'fullsize_global_accuracy': fullsize_m['global_accuracy'],
                'fullsize_per_image_accuracy': fullsize_m['per_image_accuracy'],
                'fullsize_global_vessel_iou': fullsize_m['global_vessel_iou'],
                'fullsize_global_bg_iou': fullsize_m['global_bg_iou'],
                'fullsize_global_mean_iou': fullsize_m['global_mean_iou'],
                'fullsize_per_image_vessel_iou': fullsize_m['per_image_vessel_iou'],
                'fullsize_per_image_bg_iou': fullsize_m['per_image_bg_iou'],
                'fullsize_per_image_mean_iou': fullsize_m['per_image_mean_iou'],
            })

            if fullsize_m['global_mean_iou'] > best_fullsize_global_mean_iou:
                best_fullsize_global_mean_iou = fullsize_m['global_mean_iou']
                best_epoch = epoch
                epochs_since_best = 0
                torch.save(model.model.state_dict(), env.saved_weights_filepath)
            else:
                epochs_since_best += 1
                if epochs_since_best >= config.patience:
                    wandb.run.summary['early_stopping_triggered'] = True
                    break

    except KeyboardInterrupt:
        print(f't={time.time() - start_time:.2f}: Training manually interrupted')
        wandb.run.summary['training_manually_interrupted'] = True

    finally:
        history['best']['fullsize_global_mean_iou'] = best_fullsize_global_mean_iou
        history['best']['epoch'] = best_epoch

        print()
        print('==================== Results ======================')
        print(f'Best fullsize global mean IoU: {best_fullsize_global_mean_iou:.4f}')
        print(f'Best epoch: {best_epoch}')
        print('===================================================')
        print()

        wandb.run.summary['best_fullsize_global_mean_iou'] = best_fullsize_global_mean_iou
        wandb.run.summary['best_epoch'] = best_epoch

        plot_training_metrics(history)

        with open(env.training_output_folder + 'history.json', 'w') as json_file:
            json.dump(history, json_file, indent=4)

        wandb.save(env.saved_weights_filepath)
        wandb.save(env.training_output_folder + 'history.json')

        wandb.finish()

<h2 style = "color: green;">5. Inference [10 Points]</h2>

**Plot a few samples with the test image, predicted mask and the predicted mask overlayed onto the retinal eye image.**

**For example:**

---

<img src='https://www.dropbox.com/scl/fi/re6sw5od5k59u4bwgbfgy/prediction_overlay_segformer.png?rlkey=a34frfceiq2pawoovm4zx7b5r&st=qi3fivau&dl=1'>

---



In [ ]:
@torch.no_grad()
def run_val_set_inference():
    start_time = time.time()
    print('t=0: Starting validation set inference')

    val_filenames = env.fetch_val_filenames()
    val_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.val_transforms,
        image_filenames=val_filenames,
        has_masks=True,
    )
    val_loader = create_dataloader(val_dataset, shuffle=False)

    model = RetinaSegModel(saved_model_weights=env.saved_weights_filepath)
    model.eval()

    pred_masks = []

    num_batches = num_batches_from_loader(val_loader)
    for batch_number, (x, y) in enumerate(val_loader):
        print(f't={time.time() - start_time:.2f}: Loading validation batch {batch_number + 1}/{num_batches}')

        x = x.to(env.device, non_blocking=True)
        y = y.to(env.device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=config.use_amp):
            logits = model(x)
            # logits shape: (batch, 1, H, W) → squeeze channel → (batch, H, W)
            batch_masks = (logits[:, 0] > 0).long()
            pred_masks.extend(batch_masks)

    print(f't={time.time() - start_time:.2f}: Inference complete, returning pred_masks')
    return pred_masks

In [ ]:
def visualize_pred_masks(pred_masks, num_to_visualize=3):
    val_filenames = env.fetch_val_filenames()
    val_dataset = SegmentationDataset(
        images_root_folder=env.val_images_folder,
        masks_root_folder=env.val_labels_folder,
        image_transforms=config.test_transforms,
        image_filenames=val_filenames,
        has_masks=True,
    )

    indices = random.sample(range(len(val_dataset)), num_to_visualize)

    fig, axes = plt.subplots(num_to_visualize, 3, figsize=(15, 5 * num_to_visualize))
    axes[0, 0].set_title('Image')
    axes[0, 1].set_title('Predicted Mask')
    axes[0, 2].set_title('Overlay')

    for row, i in enumerate(indices):
        image_tensor, _ = val_dataset[i]
        image = gpu_image_tensor_to_numpy_array(image_tensor)

        pred_mask = pred_masks[i].cpu().numpy() if isinstance(pred_masks[i], torch.Tensor) else np.array(pred_masks[i])
        if pred_mask.shape != (image.shape[0], image.shape[1]):
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (image.shape[1], image.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)
        colored_mask = CLASS_COLORS[(pred_mask > 0).astype(np.int32)]
        overlay = (0.5 * colored_mask + 0.5 * image).astype(np.uint8)

        axes[row, 0].imshow(image)
        axes[row, 0].axis('off')
        axes[row, 1].imshow(colored_mask)
        axes[row, 1].axis('off')
        axes[row, 2].imshow(overlay)
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
pred_masks = run_val_set_inference()

In [ ]:
global_mIoU = val_mean_iou(pred_masks)

In [ ]:
visualize_pred_masks(pred_masks)